# 9. In this exercise, we will predict the number of applications received using the other variables in the College data set.

In [5]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from sklearn.base import clone

College = pd.read_csv('College.csv', index_col=0, encoding='latin1', header=1)

if 'Private' in College.columns:
    College['Private'] = College['Private'].map({'Yes': 1, 'No': 0})

Y = College['Apps']
X = College.drop(columns=['Apps'])

scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns, index=X.index)

results = {}

## (a) Split the data set into a training set and a test set.

In [16]:
X_train, X_test, Y_train, Y_test = train_test_split(
    X_scaled, Y,
    test_size=0.3,
    random_state=42
)
print(f"訓練集大小: {len(X_train)}\n測試集大小: {len(X_test)}")

訓練集大小: 543
測試集大小: 234


## (b) Fit a linear model using least squares on the training set, and report the test error obtained.
$$ MSE = 1931803.1942$$

In [19]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

ols = LinearRegression()
ols.fit(X_train, Y_train)
Y_pred_ols = ols.predict(X_test)

test_error_ols = mean_squared_error(Y_test, Y_pred_ols)
results['OLS'] = test_error_ols

print(f"{test_error_ols:.4f}")

1931803.1942


## (c) Fit a ridge regression model on the training set, with λ chosen by cross-validation. Report the test error obtained.

$$ MSE = 1908636.4854$$

In [23]:
from sklearn.linear_model import Ridge, RidgeCV
from sklearn.metrics import mean_squared_error

alphas = 10**np.linspace(10, -2, 100) * 0.5

ridge_cv = RidgeCV(alphas=alphas, cv=5)
ridge_cv.fit(X_train, Y_train)
best_alpha_ridge = ridge_cv.alpha_

ridge = Ridge(alpha=best_alpha_ridge)
ridge.fit(X_train, Y_train)
Y_pred_ridge = ridge.predict(X_test)

test_error_ridge = mean_squared_error(Y_test, Y_pred_ridge)
results['Ridge'] = test_error_ridge

print(f"交叉驗證選擇的最佳 λ: {best_alpha_ridge:.4f}")
print(f"嶺迴歸 (Ridge) 的測試 MSE: {test_error_ridge:.4f}")

交叉驗證選擇的最佳 λ: 0.7600
嶺迴歸 (Ridge) 的測試 MSE: 1908636.4854


## (d) Fit a lasso model on the training set, with λ chosen by crossvalidation. Report the test error obtained, along with the number of non-zero coefcient estimates.

$$ MSE = 1928496.0156$$

In [22]:
from sklearn.linear_model import Lasso, LassoCV
from sklearn.metrics import mean_squared_error

lasso_cv = LassoCV(alphas=None, cv=5, max_iter=10000, random_state=42)
lasso_cv.fit(X_train, Y_train)
best_alpha_lasso = lasso_cv.alpha_

lasso = Lasso(alpha=best_alpha_lasso, max_iter=10000)
lasso.fit(X_train, Y_train)
Y_pred_lasso = lasso.predict(X_test)

test_error_lasso = mean_squared_error(Y_test, Y_pred_lasso)
results['Lasso'] = test_error_lasso

non_zero_coefs = np.sum(lasso.coef_ != 0)

print(f"交叉驗證選擇的最佳 λ: {best_alpha_lasso:.4f}")
print(f"Lasso 的測試 MSE: {test_error_lasso:.4f}")
print(f"Lasso 模型的非零係數數量: {non_zero_coefs}")

交叉驗證選擇的最佳 λ: 3.9456
Lasso 的測試 MSE: 1928496.0156
Lasso 模型的非零係數數量: 16


## (e) Fit a PCR model on the training set, with M chosen by crossvalidation. Report the test error obtained, along with the value of M selected by cross-validation.

$$ Best M = 17, MSE = 1931803.1942$$

In [24]:
from sklearn.model_selection import KFold
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

def pcr_cv(X_train, Y_train, X_test, Y_test, max_M):

    pca = PCA(n_components=max_M)
    X_train_pca = pca.fit_transform(X_train)
    X_test_pca = pca.transform(X_test)

    cv_scores = []

    kf = KFold(n_splits=5, shuffle=True, random_state=42)

    for M in range(1, max_M + 1):
        mse_folds = []
        for train_index, val_index in kf.split(X_train_pca):
            X_train_fold = X_train_pca[train_index, :M]
            Y_train_fold = Y_train.iloc[train_index]
            X_val_fold = X_train_pca[val_index, :M]

            pcr_model = LinearRegression()
            pcr_model.fit(X_train_fold, Y_train_fold)
            Y_pred_val = pcr_model.predict(X_val_fold)

            mse_folds.append(mean_squared_error(Y_train.iloc[val_index], Y_pred_val))

        cv_scores.append(np.mean(mse_folds))

    best_M = np.argmin(cv_scores) + 1

    final_pcr_model = LinearRegression()
    final_pcr_model.fit(X_train_pca[:, :best_M], Y_train)
    Y_pred_test = final_pcr_model.predict(X_test_pca[:, :best_M])

    test_error = mean_squared_error(Y_test, Y_pred_test)

    return best_M, test_error

max_components = X_train.shape[1]
best_M_pcr, test_error_pcr = pcr_cv(X_train, Y_train, X_test, Y_test, max_components)
results['PCR'] = test_error_pcr

print(f"交叉驗證選擇的最佳 M: {best_M_pcr}")
print(f"PCR 的測試 MSE: {test_error_pcr:.4f}")

交叉驗證選擇的最佳 M: 17
PCR 的測試 MSE: 1931803.1942


## (f) Fit a PLS model on the training set, with M chosen by crossvalidation. Report the test error obtained, along with the value of M selected by cross-validation.